## Import necessary library


In [ ]:
#import pandas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
!pip install -U "flwr[simulation]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.1/65.1 MB 11.6 MB/s eta 0:00:00


In [ ]:
import warnings

In [ ]:

# Suppress warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
VERBOSE = 0

In [ ]:
!pip install imbalanced-learn


In [ ]:
!pip install --upgrade numpy pandas


In [ ]:
from typing import Dict, List, Tuple

from flwr.common import Metrics


In [ ]:
from imblearn.over_sampling import SMOTE

## Preprocess and split data


In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
#Preprocessing function for one dataset
def preprocess_dataset(data_path, columns, target_column):
    df=pd.read_csv(data_path)
    #Select relevant columns
    df_cols=df[columns]
    def balance_classes_with_smote(data_df):
        features= data_df.drop(columns=target_column)
        labels=data_df[target_column]
        #Apply SMOTE to oversammple the minority class
        smote= SMOTE(random_state=1)
        features_resampled, labels_resampled = smote.fit_resample(features, labels)
        return pd.DataFrame(features_resampled,columns=features.columns), pd.Series(labels_resampled,name=target_column)
    #One-hot encoding and scaling
    df_encoded=pd.get_dummies(df_cols)
    #Apply SMOTE to handle class imbalance
    features_balanced, labels_balanced = balance_classes_with_smote(df_encoded)
    scaler=MinMaxScaler()
    features_scaled=pd.DataFrame(scaler.fit_transform(features_balanced), columns=features_balanced.columns)
    #Combine features and labels
    df_scaled=pd.concat([features_scaled, labels_balanced],axis=1)
    df_scaled = df_scaled.dropna(subset=[target_column])

    #Split into train and test
   # train_df, test_df=train_test_split(df_scaled, test_size=0.1, random_state=42)
    return df_scaled

In [ ]:
#Upload and preprocess creditcard_2023
creditcard_2023_scaled=preprocess_dataset("/content/creditcard_2023.csv",["Class","V1","V2","Amount"],'Class')

In [ ]:
train_df, test_df = train_test_split(creditcard_2023_scaled, test_size=0.1, random_state=42)

In [ ]:
# Convert to numpy arrays
train_features = train_df.drop(columns=["Class"]).values
train_labels = train_df["Class"].values
test_features = test_df.drop(columns=["Class"]).values
test_labels = test_df["Class"].values

# Combine features and labels for the training dataset
train_data = np.concatenate((train_features, train_labels.reshape(-1, 1)), axis=1)

# Combine features and labels for the test dataset
test_data = np.concatenate((test_features, test_labels.reshape(-1, 1)), axis=1)

# Ensure that data is in float32 format for TensorFlow compatibility
train_data = train_data.astype(np.float32)
test_data = test_data.astype(np.float32)

# Create Partitions for federated learning
num_partitions = 10
partitions = []
partition_size = len(train_data) // num_partitions

for i in range(num_partitions):
    start_idx = i * partition_size
    end_idx = (i + 1) * partition_size
    partition_data = train_data[start_idx:end_idx]
    partitions.append(partition_data)

NUM_CLIENTS = num_partitions

# Number of samples in train and test data
num_train_samples = train_data.shape[0]
num_test_samples = test_data.shape[0]
print(f"Number of samples in train data: {num_train_samples}")
print(f"Number of samples in test data: {num_test_samples}")

Number of samples in train data: 511767
Number of samples in test data: 56863


## Federated Learning Model setup

In [ ]:
!pip install tensorflow

  Using cached numpy-2.0.2-cp310-cp310-win_amd64.whl (15.9 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Uninstalling numpy-2.1.3:
      Successfully uninstalled numpy-2.1.3


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\Thao Tran\\anaconda3\\Lib\\site-packages\\~-mpy.libs\\libscipy_openblas64_-c16e4918366c6bc1f1cd71e28ca36fc0.dll'
Consider using the `--user` option or check the permissions.



In [ ]:
import tensorflow as tf
print(tf.__version__)

2.17.1


In [ ]:
!pip install flwr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.2/512.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.68.1
    Uninstalling grpcio-1.68.1:
      Successfully uninstalled grpcio-1.68.1
  Attempting uninstall: cryptography
    Found existing installation: cryptography 43.0.3
    Uninstalling cryptography-43.0.3:
      Successfully uninstalled cryptography-43.0.3
  Attempting uninstall: typer
    Found existing installation: typer 0.15.0
    Uninstalling typer-0.15.0:
      Successfully uninstalled typer-0.15.0


In [ ]:
import flwr as fl
print(fl.__version__)


1.13.1


In [ ]:
import tensorflow as tf
def get_model():
    """Construct a simple binary classification model"""
    model= tf.keras.models.Sequential([
        tf.keras.layers.Dense(128,activation='relu',input_shape=(train_data.shape[1]-1,)),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(1,activation='sigmoid')
    ])
    model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
    return model

In [ ]:
#FlowerClient class

from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
class FlowerClient(fl.client.NumPyClient):
    def __init__(self,model,trainset,valset) -> None:
        self.model=get_model()
        self.trainset = trainset
        self.valset= valset
    def get_parameters(self,config):
        return self.model.get_weights()

    def fit(self, parameters, config):
        self.model.set_weights(parameters)
        train_features = self.trainset[:, :-1]
        train_labels = self.trainset[:, -1]
        self.model.fit(train_features, train_labels, epochs=1, verbose=0)

        return self.model.get_weights(), len(train_features), {}

    def evaluate(self, parameters, config):

        self.model.set_weights(parameters)

        val_features = self.valset[:, :-1]  #Extract features

        val_labels = self.valset[:, -1] #Extract labels
        predictions=self.model.predict(val_features)
        predicted_labels = (predictions >0.5).astype(int)

        loss, acc = self.model.evaluate(val_features, val_labels, verbose=0)
    # Calculate precision, recall, and F1-score
        precision = precision_score(val_labels,predicted_labels)
        recall =recall_score(val_labels, predicted_labels)
        f1= f1_score(val_labels,predicted_labels)
        #Print a classification report for detailed metrics
        print(classification_report(val_labels,predicted_labels))
        #Return loss and metrics

        return loss, len(val_features), {"accuracy": acc,"precision":precision, "recall": recall,"f1-score":f1}

In [ ]:
# Weighted average for aggregation of metrics
def weighted_average(metrics: List[Tuple[int, Metrics]]) -> Metrics:
    accuracies = [num_examples * m["accuracy"] for num_examples, m in metrics]
    precisions = [num_examples * m["precision"] for num_examples, m in metrics]
    recalls = [num_examples * m["recall"] for num_examples, m in metrics]
    f1_scores = [num_examples * m["f1_score"] for num_examples, m in metrics]
    examples = [num_examples for num_examples, _ in metrics]

    # Aggregate and return custom metrics (weighted average)
    return {
        "accuracy": sum(accuracies) / sum(examples),
        "precision": sum(precisions) / sum(examples),
        "recall": sum(recalls) / sum(examples),
        "f1_score": sum(f1_scores) / sum(examples)
    }

# Function to create clients
def get_client_fn(partitions: List[np.ndarray], testset: np.ndarray):
    def client_fn(cid: str) -> fl.client.Client:
        model=get_model()
        partition = partitions[int(cid)]
        trainset, valset = partition, testset
        return FlowerClient(model,trainset, valset)
    return client_fn

def get_evaluate_fn(testset: np.ndarray):
    def evaluate(server_round: int, parameters: fl.common.NDArray, config: Dict[str, fl.common.Scalar]):
        model = get_model()
        model.set_weights(parameters)
        val_features = testset[:, :-1]
        val_labels = testset[:, -1]
        loss, accuracy = model.evaluate(val_features, val_labels, verbose=VERBOSE)

        # Add additional metrics calculations here
        predictions = (model.predict(val_features) > 0.5).astype(int)
        precision = precision_score(val_labels, predictions)
        recall = recall_score(val_labels, predictions)
        f1 = f1_score(val_labels, predictions)

        # Return aggregated metrics
        return loss, {"accuracy": accuracy, "precision": precision, "recall": recall, "f1_score": f1}
    return evaluate

In [ ]:
# Create FedAvg strategy
strategy = fl.server.strategy.FedAvg(
    fraction_fit=0.1,
    fraction_evaluate=0.05,
    min_fit_clients=10,
    min_evaluate_clients=5,
    min_available_clients=int(NUM_CLIENTS * 0.75),
    evaluate_fn=get_evaluate_fn(test_data),  # This should include precision, recall, and F1-score
    evaluate_metrics_aggregation_fn=weighted_average
)

# Define the resources each client should use
client_resources = {
    "num_cpus": 2,  # Allocate 1 CPU per client (adjust based on your system)
    "num_gpus": 0.5  # Allocate 10% of a GPU per client (if you're using GPUs)
}

# Run the Flower simulation
history = fl.simulation.start_simulation(
    client_fn=get_client_fn(partitions, test_data),
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=10),
    strategy=strategy,
    client_resources=client_resources
)

# After training, you'll have the aggregated precision, recall, and F1-score
print("Final metrics:", history)


Setting `min_available_clients` lower than `min_fit_clients` or
`min_evaluate_clients` can cause the server to fail when there are too few clients
connected to the server. `min_available_clients` must be set to a value larger
than or equal to the values of `min_fit_clients` and `min_evaluate_clients`.

Setting `min_available_clients` lower than `min_fit_clients` or
`min_evaluate_clients` can cause the server to fail when there are too few clients
connected to the server. `min_available_clients` must be set to a value larger
than or equal to the values of `min_fit_clients` and `min_evaluate_clients`.

	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
      

1777/1777 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
INFO :      initial parameters (loss, other metrics): 0.6947954297065735, {'accuracy': 0.5011870861053467, 'precision': 0.0, 'recall': 0.0, 'f1_score': 0.0}
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)
(ClientAppActor pid=6375) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.common import Context`
(ClientAppActor pid=6375) 
(ClientAppActor pid=6375)             This is a deprecated feature. It will be removed
(ClientAppActor pid=6375)             entir

1777/1777 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step


INFO :      fit progress: (1, 0.5398666858673096, {'accuracy': 0.6842762231826782, 'precision': 0.662028823108289, 'recall': 0.7498589761669722, 'f1_score': 0.7032120480732671}, 59.50159048800015)
INFO :      configure_evaluate: strategy sampled 5 clients (out of 10)
(ClientAppActor pid=6375) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.common import Context`
(ClientAppActor pid=6375) 
(ClientAppActor pid=6375)             This is a deprecated feature. It will be removed
(ClientAppActor pid=6375)             entirely in future versions of Flower.
(ClientAppActor pid=6375)         
(ClientAppActor pid=6375) WARNING :   Deprecation Warning: The `client_fn` function must return an instance of `Client`, but an instance of `NumpyClient` was returned. Please use `NumPyClient.to_client()` method to convert it to `C

...

(ClientAppActor pid=6375) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.common import Context`
(ClientAppActor pid=6375) 
(ClientAppActor pid=6375)             This is a deprecated feature. It will be removed
(ClientAppActor pid=6375)             entirely in future versions of Flower.
(ClientAppActor pid=6375)         
(ClientAppActor pid=6375) /usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
(ClientAppActor pid=6375)   super().__init__(activity_regularizer=activity_regularizer, **kwargs)
(ClientAppActor pid=6375) WARNING :   Deprecation Warning: The `client_fn` function must return an instance of `Clie

  36/1777 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step    
 105/1777 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
 174/1777 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
 240/1777 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
 305/1777 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
 371/1777 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
 428/1777 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
 489/1777 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
 550/1777 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
 610/1777 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
 676/1777 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
 735/1777 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
 793/1777 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
 859/1777 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
 923/1777 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
 991/1777 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
1054/1777 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
1119/1777 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
1150/1777 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
1211/1777 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
1276/1777 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
1335/1777 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
1395/1777 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
1459/17

INFO :      aggregate_evaluate: received 5 results and 0 failures
ERROR :     'f1_score'
ERROR :     Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/flwr/simulation/legacy_app.py", line 359, in start_simulation
    hist = run_fl(
  File "/usr/local/lib/python3.10/dist-packages/flwr/server/server.py", line 492, in run_fl
    hist, elapsed_time = server.fit(
  File "/usr/local/lib/python3.10/dist-packages/flwr/server/server.py", line 145, in fit
    res_fed = self.evaluate_round(server_round=current_round, timeout=timeout)
  File "/usr/local/lib/python3.10/dist-packages/flwr/server/server.py", line 203, in evaluate_round
    ] = self.strategy.aggregate_evaluate(server_round, results, failures)
  File "/usr/local/lib/python3.10/dist-packages/flwr/server/strategy/fedavg.py", line 281, in aggregate_evaluate
    metrics_aggregated = self.evaluate_metrics_aggregation_fn(eval_metrics)
  File "<ipython-input-15-93880e6b9ff3>", line 6, in weighted_average
    f

(ClientAppActor pid=6375)               precision    recall  f1-score   support
(ClientAppActor pid=6375) 
(ClientAppActor pid=6375)          0.0       0.71      0.62      0.66     28499
(ClientAppActor pid=6375)          1.0       0.66      0.75      0.70     28364
(ClientAppActor pid=6375) 
(ClientAppActor pid=6375)     accuracy                           0.68     56863
(ClientAppActor pid=6375)    macro avg       0.69      0.68      0.68     56863
(ClientAppActor pid=6375) weighted avg       0.69      0.68      0.68     56863
(ClientAppActor pid=6375) 


RuntimeError: Simulation crashed.